# Counterfactual corrective BC: SACSoN → NWM rollouts → ViNT policy

Combines `01_mbra_reannotate_and_embed.ipynb` (MBRA relabelling) and the
counterfactual idea from `02_counterfactual_generation.ipynb`, but the
drift-then-recover frames are now **generated by a navigation world model**
([NWM / CDiT-XL](https://huggingface.co/facebook/nwm), trained on SACSoN/HuRoN +
RECON + SCAND + TartanDrive) instead of depth + point-cloud reprojection +
inpainting. NWM is action-conditioned video prediction: given 4 real context
frames + a per-step action sequence `(dx, dy, dyaw)` it rolls out the future
egocentric view. We feed it a **perturbed** action sequence to imagine the drift.

Pipeline per trajectory:
1. **MBRA relabel** every frame → `(v, ω)`; `dt` pinned from the real `position`
   path length (the HDF5 group attrs carry real `position` + `yaw`).
2. For each window: perturb the heading with a drift-then-recover curve, convert
   to NWM actions, **NWM rolls out the drifted history frames**.
3. Save **both** a `nominal` sample (real frames → real waypoints) and a
   `counterfactual` sample (NWM frames → recovery waypoints).
4. Samples are pushed to an **HF dataset repo** (`DATA_REPO`) — the data is kept,
   not discarded.
5. A **ViNT policy is trained from scratch** on the accumulated samples and
   pushed to an **HF model repo** (`REPO_NAME`).

A Drive manifest tracks processed trajectories so generation + training span
sessions. `SACSON/HuRoN` note: NWM trained on a private higher-res version, so
expect some domain gap on the public 120×160 frames.

## 1. Setup

In [ ]:
# Colab.
!pip install -q "efficientnet-pytorch==0.7.1" "huggingface_hub>=0.24" h5py opencv-python-headless pyyaml safetensors diffusers einops timm python-dotenv

import glob
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
from collections import defaultdict

import cv2
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import yaml
from PIL import Image
from torchvision import transforms
from tqdm.auto import tqdm

from google.colab import drive
drive.mount("/content/drive")

MBRA_REPO = "/content/Learning-to-Drive-Anywhere-with-MBRA"
if not os.path.exists(MBRA_REPO):
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/vaishnavi-suresh/Learning-to-Drive-Anywhere-with-MBRA.git", MBRA_REPO], check=True)
sys.path.insert(0, os.path.join(MBRA_REPO, "train"))

NWM_REPO = "/content/nwm"
if not os.path.exists(NWM_REPO):
    subprocess.run(["git", "clone", "-q", "https://github.com/facebookresearch/nwm.git", NWM_REPO], check=True)

from vint_train.models.exaug.exaug import ExAug_dist_delay  # noqa: F401  (prereq check)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu"
GPU_MEM_GB = (torch.cuda.get_device_properties(0).total_memory / 1e9) if device.type == "cuda" else 0.0
cv2.setNumThreads(max(1, os.cpu_count() or 2))
torch.backends.cudnn.benchmark = True
print("device:", device, "|", GPU_NAME, "| %.0f GB" % GPU_MEM_GB)

## 2. Config

In [ ]:
# --- Dataset ---
HF_DATASET = "konpat/huron-sacson-hdf5"
H5_FILE = "sacson.h5"
MAX_TRAJ = None          # None = whole /train split (~2067). Resumable via the Drive manifest.
N_VAL_TRAJ = 80          # /test trajectories reserved for validation
SAMPLE_STRIDE = 5        # overlapping window stride (frames)
MAX_SAMPLES_PER_TRAJ = 8
METRIC_WAYPOINT_SPACING = 0.255   # sacson value from vint_train/data/data_config.yaml (NWM action normalization)
CALIBRATE_DT_FROM_POSITION = True
SACSON_FPS = 3.0                  # fallback dt = 1/SACSON_FPS

# --- MBRA (fixed by the checkpoint) ---
CONTEXT_SIZE = 5
LEN_TRAJ_PRED = 8
ROBOT_SIZE, DELAY = 0.3, 0.0
VEL_PAST_LINEAR, VEL_PAST_ANGULAR = 0.5, 0.0

# --- Counterfactual window ---
TH_HISTORY = 10          # history frames in a window (obs = last CONTEXT_SIZE+1 of these)
T_FUTURE = 15            # recovery frames after t
WINDOW_N = TH_HISTORY + T_FUTURE
GOAL_HORIZON = 15        # real frame this far past t is the ViNT goal image
MBRA_GOAL_HORIZON = 15
PERTURBATION_LEVELS = {"mild": 0.15, "strong": 0.25}   # peak heading deviation (rad)

# --- NWM (navigation world model) ---
NWM_SIZE = "xl"          # s | b | l | xl  (facebook/nwm checkpoints)
NWM_CONTEXT = 4          # CDiT context frames (fixed by the checkpoint)
NWM_IMG = 224            # CDiT native resolution
NWM_DDIM_STEPS = 0       # 0 -> exact repo config: 250-step DDPM p_sample. >0 -> that many DDIM steps (faster, lower fidelity)
NWM_BATCH = 16 if GPU_MEM_GB >= 30 else (8 if GPU_MEM_GB >= 20 else 4)   # windows rolled out together
NWM_FP16 = True          # autocast bf16 where supported (T4/V100 fall back to fp32)
NWM_ACTION_MIN = [-2.5, -4.0]   # nwm/vint_train data_config action_stats -- (dx, dy) normalization
NWM_ACTION_MAX = [5.0, 4.0]

# --- ViNT policy / training ---
IMAGE_SIZE = (96, 96)
VINT_OBS_ENCODING_SIZE = 512
VINT_MHA = 4
BATCH_SIZE = 128
EPOCHS = 80
LR = 3e-4
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 4
ALPHA = 0.5
WAYPOINT_SCALE = 1.0
AUG = True
USE_AMP = True

# --- Paths / publishing ---
DRIVE_ROOT = "/content/drive/MyDrive/sacson_nwm_bc"
CACHE_DIR = "/content/bc_cache"                 # local working copy of the sample cache
MANIFEST_PATH = os.path.join(DRIVE_ROOT, "processed_manifest.json")
CKPT_DIR = os.path.join(DRIVE_ROOT, "checkpoints")
CLIP_DIR = os.path.join(DRIVE_ROOT, "counterfactual_clips")
SAVE_CLIPS_FOR_FIRST_N = 10
DATA_REPO = "vint-sacson-bc-data"              # HF dataset repo (samples are pushed here, kept)
REPO_NAME = "vint-sacson-corrective"           # HF model repo
PUSH_DATA_EVERY = 20                            # push new npz to the HF dataset repo every N trajectories
RESUME_FROM_HF = True
SEED = 0

for d in (DRIVE_ROOT, CACHE_DIR, CKPT_DIR, CLIP_DIR,
          os.path.join(CACHE_DIR, "train"), os.path.join(CACHE_DIR, "test")):
    os.makedirs(d, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# --- HF token: env var -> .env (same machine only) -> interactive prompt -> verify ---
ENV_FILE = ""          # optional: absolute path to a .env ON THIS MACHINE (Colab != your laptop)
from dotenv import load_dotenv, find_dotenv
from huggingface_hub import HfApi
import getpass


def _clean_tok():
    return (os.environ.get("HF_TOKEN") or "").strip().strip("'\"")


if not _clean_tok():
    _p = ENV_FILE or find_dotenv(usecwd=True)
    if _p and os.path.exists(_p):
        load_dotenv(_p, override=True)
        print("dotenv:", _p)

while True:
    tok = _clean_tok()
    if not tok:
        tok = getpass.getpass("Paste your Hugging Face token (write scope, hidden): ").strip()
    os.environ["HF_TOKEN"] = tok
    try:
        HF_USER = HfApi().whoami(token=tok)["name"]
        print("HF logged in as:", HF_USER)
        break
    except Exception as e:
        print("token rejected (%s) -- try again" % type(e).__name__)
        os.environ.pop("HF_TOKEN", None)


def load_hf_token():
    tok = _clean_tok()
    if not tok:
        raise RuntimeError("HF_TOKEN not set -- rerun the Section 2 config cell (it will prompt).")
    return tok


print("NWM batch:", NWM_BATCH, "| window:", WINDOW_N,
      "| min traj length:", CONTEXT_SIZE + (TH_HISTORY - 1) + max(T_FUTURE, GOAL_HORIZON) + 2)

## 3. Fetch the HDF5, inspect schema, helpers

The trajectory group attrs carry `num_frames`, real `position` (N,2) metres, real
`yaw` (N,) rad, `trajectory_name`. Frames are vlen JPEG bytes (~120×160).

In [ ]:
from huggingface_hub import hf_hub_download

H5_PATH = hf_hub_download(HF_DATASET, H5_FILE, repo_type="dataset", token=load_hf_token())
print("hdf5:", H5_PATH, "(%.2f GB)" % (os.path.getsize(H5_PATH) / 1e9))

_h5 = h5py.File(H5_PATH, "r")
print("groups:", list(_h5.keys()))
for split in [s for s in ("train", "test") if s in _h5]:
    keys = list(_h5[split].keys())
    node = _h5[split][keys[0]]
    print("/%s: %d trajectories | sample %r" % (split, len(keys), keys[0]))
    if isinstance(node, h5py.Group):
        for name, item in node.items():
            print("   %s: %s %s" % (name, getattr(item, "shape", None), getattr(item, "dtype", "")))
        print("   attr keys:", list(node.attrs))


def _decode_jpeg(buf):
    b = buf.tobytes() if hasattr(buf, "tobytes") else bytes(buf)
    img = cv2.imdecode(np.frombuffer(b, np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError("cv2.imdecode returned None")
    return img


def load_traj(split, key):
    node = _h5[split][key]
    raw = node["frames"][()] if isinstance(node, h5py.Group) else node[()]
    return [_decode_jpeg(b) for b in raw]


def traj_odom(split, key):
    a = _h5[split][key].attrs
    return np.asarray(a["position"], dtype=np.float64), np.asarray(a["yaw"], dtype=np.float64)


def wrap_pi(a):
    return (a + np.pi) % (2 * np.pi) - np.pi


_p = load_traj("train", list(_h5["train"].keys())[0])
_pp, _py = traj_odom("train", list(_h5["train"].keys())[0])
print("load_traj: %d frames %s | odom: pos %s yaw %s" % (len(_p), _p[0].shape, _pp.shape, _py.shape))

## 4. MBRA model + `label_traj`

In [ ]:
with open(os.path.join(MBRA_REPO, "train", "config", "MBRA.yaml")) as f:
    mbra_config = yaml.safe_load(f)
assert mbra_config["context_size"] == CONTEXT_SIZE and mbra_config["len_traj_pred"] == LEN_TRAJ_PRED

mbra = ExAug_dist_delay(
    context_size=mbra_config["context_size"], len_traj_pred=mbra_config["len_traj_pred"],
    learn_angle=mbra_config["learn_angle"], obs_encoder=mbra_config["obs_encoder"],
    obs_encoding_size=mbra_config["obs_encoding_size"], late_fusion=mbra_config["late_fusion"],
    mha_num_attention_heads=mbra_config["mha_num_attention_heads"],
    mha_num_attention_layers=mbra_config["mha_num_attention_layers"],
    mha_ff_dim_factor=mbra_config["mha_ff_dim_factor"])
_w = os.path.join(MBRA_REPO, "deployment", "mbra.pth")
if not os.path.exists(_w):
    os.makedirs(os.path.dirname(_w), exist_ok=True)
    shutil.copy(hf_hub_download("NHirose/MBRA_project_models", "mbra.pth"), _w)
_miss, _ = mbra.load_state_dict(torch.load(_w, map_location=device), strict=False)
assert not _miss, "missing MBRA keys: %s" % _miss
mbra.eval().to(device)

frame_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE[1], IMAGE_SIZE[0])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])


def to_rgb(bgr):
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def frame_tensor(bgr):
    return frame_transform(Image.fromarray(to_rgb(bgr)))


_robot_size = ROBOT_SIZE * torch.ones(1, 1, 1, device=device)
_delay = DELAY * torch.ones(1, 1, 1, device=device)
_vel_past = torch.cat([VEL_PAST_LINEAR * torch.ones(1, 6),
                       VEL_PAST_ANGULAR * torch.ones(1, 6)], dim=1).unsqueeze(2).to(device)


def integrate_poses(v, w, dt):
    """MBRA (v, omega) + per-step dt -> (x, y, theta) dead-reckoned track."""
    n = len(v)
    poses = np.zeros((n, 3))
    x = y = th = 0.0
    for i in range(n):
        th += w[i] * dt[i]
        x += v[i] * math.cos(th) * dt[i]
        y += v[i] * math.sin(th) * dt[i]
        poses[i] = (x, y, th)
    return poses


def label_traj(split, key):
    frames = load_traj(split, key)
    n = len(frames)
    if n < CONTEXT_SIZE + (TH_HISTORY - 1) + max(T_FUTURE, GOAL_HORIZON) + 2:
        return None
    ftens = [frame_tensor(f) for f in frames]
    hi = n - max(T_FUTURE, GOAL_HORIZON) - 1
    rows = []
    for t in range(CONTEXT_SIZE, hi):
        obs = torch.cat(ftens[t - CONTEXT_SIZE:t + 1], dim=0).unsqueeze(0).to(device)
        goal = ftens[min(t + MBRA_GOAL_HORIZON, n - 1)].unsqueeze(0).to(device)
        with torch.no_grad():
            lin, ang, _ = mbra(obs, goal, _robot_size, _delay, _vel_past)
        rows.append({"frame_id": t, "v": float(lin[0, 0]), "w": float(ang[0, 0])})
    if len(rows) < WINDOW_N:
        return None
    df = pd.DataFrame(rows)
    fids = df["frame_id"].to_numpy()
    sum_v = float(np.abs(df["v"]).sum())
    if CALIBRATE_DT_FROM_POSITION:
        pos, _ = traj_odom(split, key)
        path_len = float(np.hypot(*np.diff(pos[fids[0]:fids[-1] + 1], axis=0).T).sum())
        dt = float(np.clip(path_len / max(sum_v, 1e-6), 0.03, 3.0))
    else:
        dt = 1.0 / SACSON_FPS
    v = df["v"].to_numpy(); w = df["w"].to_numpy()
    poses = integrate_poses(v, w, np.full(len(df), dt))
    med_step = float(np.median(np.abs(v))) * dt
    return frames, fids, v, w, dt, poses, med_step

## 5. Load NWM (CDiT + SD-VAE)

Downloads `facebook/nwm` (`cdit_<size>`), builds the CDiT from the repo's
`models.py`, loads the EMA weights, and the `stabilityai/sd-vae-ft-ema`
autoencoder.

In [ ]:
sys.path.insert(0, NWM_REPO)
from models import CDiT_models          # noqa: E402
from diffusion import create_diffusion  # noqa: E402
from diffusers.models import AutoencoderKL  # noqa: E402

_NWM_CKPT = {"s": "cdit_s_100000.pth.tar", "b": "cdit_b_100000.pth.tar",
             "l": "cdit_l_100000.pth.tar", "xl": "cdit_xl_100000.pth.tar"}
_NWM_MODEL = {"s": "CDiT-S/2", "b": "CDiT-B/2", "l": "CDiT-L/2", "xl": "CDiT-XL/2"}
_NWM_LATENT = NWM_IMG // 8            # SD-VAE 8x downsample -> 28 for 224
VAE_SCALE = 0.18215

# Weights stay fp32; the speedup is torch.autocast(bfloat16) -- matches NWM's own
# inference. Hard .half() breaks the DiT embedders (they upcast to fp32), and
# fp16 autocast gives colour casts, so we use bf16 or fall back to fp32.
NWM_AC_DTYPE = torch.bfloat16 if (device.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float32
NWM_AUTOCAST = NWM_FP16 and device.type == "cuda" and NWM_AC_DTYPE == torch.bfloat16

vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-ema").to(device).eval()
for p in vae.parameters():
    p.requires_grad_(False)

_ckpt_path = hf_hub_download("facebook/nwm", _NWM_CKPT[NWM_SIZE], token=load_hf_token())
nwm = CDiT_models[_NWM_MODEL[NWM_SIZE]](input_size=_NWM_LATENT, context_size=NWM_CONTEXT, in_channels=4)
_sd = torch.load(_ckpt_path, map_location="cpu", weights_only=False)
nwm.load_state_dict(_sd["ema"] if "ema" in _sd else _sd.get("model", _sd), strict=True)
nwm = nwm.to(device).eval()
for p in nwm.parameters():
    p.requires_grad_(False)

if NWM_DDIM_STEPS and NWM_DDIM_STEPS > 0:
    nwm_diffusion = create_diffusion(timestep_respacing="ddim%d" % NWM_DDIM_STEPS)
    nwm_sample = nwm_diffusion.ddim_sample_loop
else:
    nwm_diffusion = create_diffusion(str(250))   # the repo default: 250-step DDPM p_sample
    nwm_sample = nwm_diffusion.p_sample_loop
NWM_VER = "v3-pixelctx-actionstats-bf16"   # bump when the rollout code changes; printed so you can confirm the live version
print("NWM %s loaded [%s] (%s, latent %d, autocast=%s dtype=%s), sampler=%s"
      % (NWM_SIZE, NWM_VER, _NWM_MODEL[NWM_SIZE], _NWM_LATENT, NWM_AUTOCAST,
         NWM_AC_DTYPE, "ddim%d" % NWM_DDIM_STEPS if NWM_DDIM_STEPS else "ddpm250"))

## 6. Action encoding + NWM rollout

`nwm_deltas` reproduces `nwm/datasets._compute_actions`: cumulative local
displacement, `/METRIC_WAYPOINT_SPACING`, `normalize_data(·, ACTION_STATS)` to
[-1, 1] (xy only; yaw raw), then `get_delta_np` to per-step `(dx, dy, dyaw)`.
`nwm_rollout` matches `isolated_nwm_infer.generate_rollout`: one per-step delta,
`rel_t = 1/128`, context updated after each frame, bf16 autocast.

In [ ]:
def _yaw_rotmat(yaw):
    c, s = math.cos(yaw), math.sin(yaw)
    return np.array([[c, -s], [s, c]])


def perturb_track(poses, alpha):
    """poses (WINDOW_N,3) x,y,theta -> perturbed track with a drift-then-recover
    heading offset (0 at both ends, +alpha at frame t = index TH_HISTORY-1),
    positions re-walked with the real per-step lengths."""
    n = len(poses)
    hist = np.arange(TH_HISTORY)
    dev = np.empty(n)
    dev[:TH_HISTORY] = alpha * 0.5 * (1 - np.cos(np.pi * hist / (TH_HISTORY - 1)))
    fut = np.arange(TH_HISTORY, n) - (TH_HISTORY - 1)
    dev[TH_HISTORY:] = alpha * 0.5 * (1 + np.cos(np.pi * fut / (n - TH_HISTORY)))
    yaw_p = poses[:, 2] + dev
    step = np.hypot(*np.diff(poses[:, :2], axis=0).T)
    xy = np.zeros((n, 2)); xy[0] = poses[0, :2]
    for i in range(1, n):
        xy[i] = xy[i - 1] + step[i - 1] * np.array([math.cos(yaw_p[i - 1]), math.sin(yaw_p[i - 1])])
    return np.concatenate([xy, yaw_p[:, None]], axis=1)


_A_MIN = np.asarray(NWM_ACTION_MIN, dtype=np.float32)
_A_MAX = np.asarray(NWM_ACTION_MAX, dtype=np.float32)


def nwm_deltas(track):
    """track (M,3) x,y,theta with frame 0 = last NWM context frame ->
    (M-1, 3) per-step NWM actions. Matches nwm/datasets._compute_actions +
    normalize_data(ACTION_STATS) + get_delta_np."""
    xy, yaw = track[:, :2], track[:, 2]
    cum_xy = (xy - xy[0]).dot(_yaw_rotmat(yaw[0]))                 # cumulative local displacement (frame-0 local)
    cum_yaw = wrap_pi(yaw - yaw[0])
    cum = np.concatenate([cum_xy, cum_yaw[:, None]], axis=1)[1:].astype(np.float32)   # drop frame 0
    cum[:, :2] = ((cum[:, :2] / METRIC_WAYPOINT_SPACING) - _A_MIN) / (_A_MAX - _A_MIN) * 2.0 - 1.0
    return np.diff(np.concatenate([np.zeros((1, 3), np.float32), cum], axis=0), axis=0).astype(np.float32)


def _ac():
    return torch.autocast(device_type=device.type, dtype=NWM_AC_DTYPE, enabled=NWM_AUTOCAST)


def ctx_pixels(imgs_bgr):
    """list of BGR frames -> (N, 3, NWM_IMG, NWM_IMG) tensor in [-1, 1] (VAE input range)."""
    x = np.stack([cv2.resize(to_rgb(b), (NWM_IMG, NWM_IMG)) for b in imgs_bgr]).astype(np.float32)
    return torch.from_numpy(x).permute(0, 3, 1, 2) / 127.5 - 1.0


@torch.no_grad()
def nwm_rollout(ctx_px, step_deltas, n_gen):
    """Mirrors nwm/isolated_nwm_infer.generate_rollout + model_forward_wrapper:
    pixel context, re-encoded every step; one per-step delta; rel_t = 1/128.
    ctx_px (B, NWM_CONTEXT, 3, IMG, IMG) in [-1,1]; step_deltas (B, n_gen, 3).
    Returns (B, n_gen, IMG, IMG, 3) RGB uint8."""
    B = ctx_px.shape[0]
    cur = ctx_px.to(device).float()                              # pixel context, [-1,1]
    rel_t = torch.full((B,), 1.0 / 128.0, device=device)
    outs = []
    for m in range(n_gen):
        y = torch.as_tensor(step_deltas[:, m], device=device, dtype=torch.float32)
        z = torch.randn(B, 4, _NWM_LATENT, _NWM_LATENT, device=device)
        with _ac():
            x_cond = (vae.encode(cur.flatten(0, 1)).latent_dist.sample() * VAE_SCALE).unflatten(0, (B, NWM_CONTEXT))
            samp = nwm_sample(nwm.forward, z.shape, z, clip_denoised=False, progress=False, device=device,
                              model_kwargs=dict(y=y, x_cond=x_cond.float(), rel_t=rel_t))
            px = vae.decode(samp.float() / VAE_SCALE).sample.clamp(-1, 1).float()    # (B, 3, IMG, IMG)
        outs.append(px)
        cur = torch.cat([cur[:, 1:], px.unsqueeze(1)], dim=1)     # slide pixel context
    stk = torch.stack(outs, dim=1)                                # (B, n_gen, 3, IMG, IMG)
    return ((stk + 1) * 127.5).clamp(0, 255).permute(0, 1, 3, 4, 2).cpu().numpy().astype(np.uint8)

## 7. `traj_to_bc_samples` — window → (nominal, counterfactual) pairs

`nominal` keeps the **real** photographed obs frames; `counterfactual` uses the
NWM rollout of the drifted history. Same real goal frame. Labels are cumulative
local waypoints `(x, y, cosΔθ, sinΔθ)` — real for nominal, perturbed-recovery for
the counterfactual.

In [ ]:
STEP_OFFSETS = np.linspace(1, T_FUTURE, LEN_TRAJ_PRED).round().astype(int)


def waypoints_local(track, t_idx):
    x0, y0, th0 = track[t_idx]
    c, s = math.cos(th0), math.sin(th0)
    out = np.zeros((LEN_TRAJ_PRED, 4), dtype=np.float32)
    for k, off in enumerate(STEP_OFFSETS):
        px, py, pth = track[t_idx + off]
        dx, dy = px - x0, py - y0
        out[k] = ((c * dx + s * dy) * WAYPOINT_SCALE, (-s * dx + c * dy) * WAYPOINT_SCALE,
                  math.cos(pth - th0), math.sin(pth - th0))
    return out


def _img96(rgb):
    return cv2.resize(rgb, IMAGE_SIZE, interpolation=cv2.INTER_AREA).astype(np.uint8)


def _save(split, key, t, kind, obs_rgb96, goal_rgb96, action, dist):
    d = os.path.join(CACHE_DIR, split, key)
    os.makedirs(d, exist_ok=True)
    np.savez_compressed(os.path.join(d, "%05d_%s.npz" % (t, kind)),
                        obs=np.stack(obs_rgb96).astype(np.uint8), goal=goal_rgb96.astype(np.uint8),
                        action=action.astype(np.float32), dist=np.float32(dist),
                        kind=kind, key=key, t=np.int64(t))


N_GEN = TH_HISTORY - NWM_CONTEXT          # drifted frames NWM must generate per window
assert N_GEN >= CONTEXT_SIZE + 1, "need TH_HISTORY - NWM_CONTEXT >= CONTEXT_SIZE + 1"


def traj_to_bc_samples(split, key, level, save_clip):
    got = label_traj(split, key)
    if got is None:
        return 0, 0, 0.0
    frames, fids, v, w, dt, poses, med_step = got
    n_frames = len(frames)
    alpha = PERTURBATION_LEVELS[level]
    n_rows = len(fids)

    row_is = list(range(TH_HISTORY - 1, n_rows - max(T_FUTURE, GOAL_HORIZON), SAMPLE_STRIDE))
    if len(row_is) > MAX_SAMPLES_PER_TRAJ:
        row_is = [row_is[i] for i in np.linspace(0, len(row_is) - 1, MAX_SAMPLES_PER_TRAJ).round().astype(int)]

    jobs = []
    for row_i in row_is:
        wr = list(range(row_i - TH_HISTORY + 1, row_i + T_FUTURE + 1))          # WINDOW_N rows
        wfids = fids[wr]
        t_frame = int(wfids[TH_HISTORY - 1])
        if int(wfids.max()) >= n_frames or t_frame + GOAL_HORIZON >= n_frames:
            continue
        real_track = poses[wr]
        pert_track = perturb_track(real_track, alpha)
        ctx_ids = [int(x) for x in wfids[:NWM_CONTEXT]]
        deltas = nwm_deltas(pert_track[NWM_CONTEXT - 1:TH_HISTORY])   # (N_GEN, 3), origin = last context frame
        jobs.append(dict(t=t_frame, wfids=wfids, real_track=real_track, pert_track=pert_track,
                         ctx_ids=ctx_ids, deltas=deltas,
                         goal=_img96(to_rgb(frames[t_frame + GOAL_HORIZON]))))
    if not jobs:
        return 0, 0, med_step

    n_nom = n_pert = 0
    for b0 in range(0, len(jobs), NWM_BATCH):
        batch = jobs[b0:b0 + NWM_BATCH]
        ctx_px = torch.stack([ctx_pixels([frames[i] for i in j["ctx_ids"]]) for j in batch])  # (B, ctx, 3, IMG, IMG)
        step_deltas = np.stack([j["deltas"] for j in batch])                                  # (B, N_GEN, 3)
        gen = nwm_rollout(ctx_px, step_deltas, N_GEN)                                          # (B, N_GEN, IMG, IMG, 3)
        for j, g in zip(batch, gen):
            t = j["t"]
            obs_ids = [int(x) for x in j["wfids"][TH_HISTORY - 1 - CONTEXT_SIZE: TH_HISTORY]]
            _save(split, key, t, "nominal",
                  [_img96(to_rgb(frames[i])) for i in obs_ids], j["goal"],
                  waypoints_local(j["real_track"], TH_HISTORY - 1), GOAL_HORIZON)
            n_nom += 1
            _save(split, key, t, "counterfactual",
                  [_img96(g[i]) for i in range(N_GEN - CONTEXT_SIZE - 1, N_GEN)], j["goal"],
                  waypoints_local(j["pert_track"], TH_HISTORY - 1), GOAL_HORIZON)
            n_pert += 1
            if save_clip and t == jobs[0]["t"]:
                cd = os.path.join(CLIP_DIR, key, level, "t%d" % t)
                os.makedirs(cd, exist_ok=True)
                for i in range(N_GEN):
                    cv2.imwrite(os.path.join(cd, "gen_%02d.jpg" % i), cv2.cvtColor(g[i], cv2.COLOR_RGB2BGR))
                for i, fid in enumerate(j["ctx_ids"]):
                    cv2.imwrite(os.path.join(cd, "ctx_%02d.jpg" % i), frames[fid])
                with open(os.path.join(cd, "meta.json"), "w") as fh:
                    json.dump({"key": key, "level": level, "alpha": alpha, "anchor_frame_id": t,
                               "dt": dt, "median_step_m": med_step,
                               "pert_track": j["pert_track"].tolist()}, fh)
    return n_nom, n_pert, med_step

## 8. Processing driver + push samples to the HF dataset repo

Resumable over the whole `/train` split. New `.npz` files are pushed to
`<user>/DATA_REPO` (an HF **dataset** repo) every `PUSH_DATA_EVERY` trajectories,
so the data accumulates and survives the session.

In [ ]:
from huggingface_hub import HfApi, create_repo, CommitOperationAdd

HF_TOKEN = load_hf_token()          # HF_USER was set + verified in Section 2
DATA_REPO_ID = "%s/%s" % (HF_USER, DATA_REPO)
create_repo(DATA_REPO_ID, repo_type="dataset", private=True, exist_ok=True, token=HF_TOKEN)
_hf = HfApi()


def load_manifest():
    if os.path.exists(MANIFEST_PATH):
        with open(MANIFEST_PATH) as f:
            return json.load(f)
    return {}


def save_manifest(m):
    tmp = MANIFEST_PATH + ".tmp"
    with open(tmp, "w") as f:
        json.dump(m, f, indent=1)
    os.replace(tmp, MANIFEST_PATH)


def push_new_npz(uploaded):
    files = [p for p in glob.glob(os.path.join(CACHE_DIR, "**", "*.npz"), recursive=True) if p not in uploaded]
    if not files:
        return
    ops = [CommitOperationAdd(path_in_repo=os.path.relpath(p, CACHE_DIR), path_or_fileobj=p) for p in files]
    _hf.create_commit(DATA_REPO_ID, repo_type="dataset", token=HF_TOKEN, operations=ops,
                      commit_message="add %d samples" % len(files))
    uploaded.update(files)
    print("   pushed %d npz -> %s" % (len(files), DATA_REPO_ID))


def run_processing():
    manifest = load_manifest()
    rng = random.Random(SEED)
    uploaded = set()
    plan = [("test", k) for k in sorted(_h5["test"].keys())[:N_VAL_TRAJ]]
    tk = sorted(_h5["train"].keys())
    plan += [("train", k) for k in (tk[:MAX_TRAJ] if MAX_TRAJ else tk)]

    tot_n = tot_p = 0
    steps = []
    for i, (split, key) in enumerate(tqdm(plan, desc="trajectories")):
        mkey = "%s/%s" % (split, key)
        if mkey in manifest:
            continue
        level = rng.choice(list(PERTURBATION_LEVELS))
        n_done = len([k for k in manifest if "error" not in manifest[k]])  # clips for the first N ok trajectories, any split
        try:
            n_nom, n_pert, med = traj_to_bc_samples(split, key, level,
                                                    save_clip=(n_done < SAVE_CLIPS_FOR_FIRST_N))
        except Exception as e:
            print("  %s: skipped (%s: %s)" % (mkey, type(e).__name__, e))
            manifest[mkey] = {"error": str(e)}; save_manifest(manifest); continue
        manifest[mkey] = {"split": split, "level": level, "n_nominal": n_nom,
                          "n_counterfactual": n_pert, "median_step_m": round(med, 4)}
        save_manifest(manifest)
        tot_n += n_nom; tot_p += n_pert
        if med > 0:
            steps.append(med)
        if (i + 1) % PUSH_DATA_EVERY == 0:
            push_new_npz(uploaded)
        if (i + 1) % 25 == 0:
            ms = float(np.median(steps)) if steps else float("nan")
            print("  [%d/%d] %d nominal + %d cf | median step %.3f m (ref %.3f)"
                  % (i + 1, len(plan), tot_n, tot_p, ms, METRIC_WAYPOINT_SPACING))
    push_new_npz(uploaded)
    n = len(glob.glob(os.path.join(CACHE_DIR, "**", "*.npz"), recursive=True))
    print("done. +%d samples this session; %d npz cached; data repo %s" % (tot_n + tot_p, n, DATA_REPO_ID))


run_processing()

## 8b. Inspect the NWM counterfactuals

`show_clip()` renders the saved rollouts (context → NWM-generated drift, with the
real un-drifted frames underneath for comparison) and writes an mp4.
`show_pair()` shows a cached `nominal` vs `counterfactual` `.npz` at ViNT
resolution with the waypoint labels. Auto-saved clips live in
`DRIVE_ROOT/counterfactual_clips/<key>/<level>/t<t>/` (first `SAVE_CLIPS_FOR_FIRST_N`
train trajectories).

In [ ]:
from IPython.display import Video, display


def diag():
    print("NWM rollout ver   :", globals().get("NWM_VER", "<Section 5 not run>"),
          "| autocast dtype:", globals().get("NWM_AC_DTYPE", "?"))
    print("Drive mounted     :", os.path.ismount("/content/drive") or os.path.exists("/content/drive/MyDrive"))
    print("DRIVE_ROOT exists  :", os.path.isdir(DRIVE_ROOT), "->", DRIVE_ROOT)
    print("manifest exists    :", os.path.exists(MANIFEST_PATH))
    m = load_manifest() if os.path.exists(MANIFEST_PATH) else {}
    errs = [k for k in m if "error" in m[k]]
    print("manifest entries   : %d  (%d ok, %d errored)" % (len(m), len(m) - len(errs), len(errs)))
    if errs:
        print("  first error      :", m[errs[0]]["error"][:200])
    print("clips on Drive     :", len(glob.glob(os.path.join(CLIP_DIR, "*", "*", "t*"))))
    for s in ("train", "test"):
        print("npz cached (%s)  : %d" % (s, len(glob.glob(os.path.join(CACHE_DIR, s, "**", "*.npz"), recursive=True))))


diag()


@torch.no_grad()
def nwm_selftest(split="test", which=0):
    """Isolates where a bad rollout comes from:
      row 1  real frames 4..9
      row 2  NWM given ZERO actions -> should ~freeze on the context
      row 3  NWM given the REAL odometry actions -> should ~reconstruct row 1
    If row 3 != row 1, it's the model/VAE/domain, not the perturbation."""
    key = sorted(_h5[split].keys())[which]
    frs = load_traj(split, key)
    pos, yaw = traj_odom(split, key)
    ctx = torch.stack([ctx_pixels(frs[3:7])]).to(device)                       # (1, 4, 3, IMG, IMG)
    with _ac():
        lat = vae.encode(ctx.flatten(0, 1)).latent_dist.sample() * VAE_SCALE
        rec = vae.decode(lat / VAE_SCALE).sample.clamp(-1, 1)
    recon_l1 = (rec.float().cpu() - ctx.flatten(0, 1).float().cpu()).abs().mean().item()

    real_track = np.concatenate([pos[3:10], yaw[3:10, None]], axis=1).astype(np.float64)
    d_real = nwm_deltas(real_track)[None]                                      # (1, 6, 3)
    z0 = nwm_rollout(ctx, np.zeros((1, 6, 3), np.float32), 6)[0]
    zr = nwm_rollout(ctx, d_real, 6)[0]
    real6 = [cv2.resize(to_rgb(frs[i]), (NWM_IMG, NWM_IMG)) for i in range(4, 10)]

    fig, ax = plt.subplots(3, 6, figsize=(13, 6.5))
    for i in range(6):
        for r, im in enumerate((real6[i], z0[i], zr[i])):
            ax[r][i].imshow(im); ax[r][i].axis("off")
    for r, lab in enumerate(("real 4..9", "NWM zero-action", "NWM real-action")):
        ax[r][0].set_ylabel(lab, rotation=0, ha="right", va="center", fontsize=9)
    fig.suptitle("%s  |  VAE recon L1 = %.3f (want < ~0.05)  |  %s" % (key, recon_l1, NWM_VER))
    plt.tight_layout(); plt.show(); plt.close(fig)
    print("per-step real deltas (mean abs):", np.abs(d_real[0]).mean(0))


def peek_now(split="train", level="strong", which=0):
    """Reprocess ONE trajectory right now (no manifest write), wiping its old
    clip first, so you eyeball the CURRENT code's output rather than a cached jpg."""
    k = sorted(_h5[split].keys())[which]
    shutil.rmtree(os.path.join(CLIP_DIR, k), ignore_errors=True)
    print("reprocessing", split, k, "...")
    print("  ->", traj_to_bc_samples(split, k, level, save_clip=True))
    made = sorted(glob.glob(os.path.join(CLIP_DIR, k, "*", "t*")))
    print("  clip:", made[:1])
    for cd in made[:1]:
        show_clip(only=cd)


def show_clip(n=3, save_mp4=True, fps=4, only=None):
    dirs = [only] if only else sorted(glob.glob(os.path.join(CLIP_DIR, "*", "*", "t*")))
    if not dirs or not dirs[0]:
        print("no clips under", CLIP_DIR, "-- run Section 8 with SAVE_CLIPS_FOR_FIRST_N > 0, or peek_now()")
        return
    for cd in ([only] if only else dirs[:n]):
        meta = json.load(open(os.path.join(cd, "meta.json")))
        ctx = [cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB) for p in sorted(glob.glob(cd + "/ctx_*.jpg"))]
        gen = [cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB) for p in sorted(glob.glob(cd + "/gen_*.jpg"))]
        H, W = gen[0].shape[:2]
        real = None
        try:                                    # real frames at the same window positions (frames 4..TH_HISTORY-1)
            frs = load_traj("train", meta["key"])
            gen_ids = list(range(meta["anchor_frame_id"] - len(gen) + 1, meta["anchor_frame_id"] + 1))
            real = [cv2.resize(cv2.cvtColor(frs[i], cv2.COLOR_BGR2RGB), (W, H)) for i in gen_ids]
        except Exception:
            pass

        rows = [("context (real)", ctx), ("NWM drift (gen)", gen)] + ([("real (no drift)", real)] if real else [])
        cols = max(len(r[1]) for r in rows)
        fig, ax = plt.subplots(len(rows), cols, figsize=(1.8 * cols, 1.9 * len(rows)), squeeze=False)
        for r, (label, imgs) in enumerate(rows):
            for c in range(cols):
                ax[r][c].set_xticks([]); ax[r][c].set_yticks([])
                if c < len(imgs):
                    ax[r][c].imshow(imgs[c])
            ax[r][0].set_ylabel(label, rotation=0, ha="right", va="center", fontsize=9)
        fig.suptitle("%s  t=%d  %s (α=%.2f, dt=%.3fs)"
                     % (meta["key"], meta["anchor_frame_id"], meta["level"], meta["alpha"], meta.get("dt", 0)))
        plt.tight_layout(); plt.show(); plt.close(fig)

        if save_mp4:
            seq = ctx + gen
            mp4 = cd + "/rollout.mp4"
            vw = cv2.VideoWriter(mp4, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
            for im in seq:
                vw.write(cv2.cvtColor(cv2.resize(im, (W, H)), cv2.COLOR_RGB2BGR))
            vw.release()
            display(Video(mp4, embed=True, width=W * 2))


def show_pair(idx=0, split="train"):
    cfs = sorted(glob.glob(os.path.join(CACHE_DIR, split, "*", "*_counterfactual.npz")))
    if not cfs:
        print("no counterfactual .npz in", os.path.join(CACHE_DIR, split))
        return
    cf = np.load(cfs[idx])
    nom = np.load(cfs[idx].replace("_counterfactual.npz", "_nominal.npz"))
    ncol = CONTEXT_SIZE + 2
    fig, ax = plt.subplots(3, ncol, figsize=(2 * ncol, 6),
                           gridspec_kw={"height_ratios": [1, 1, 1.3]})
    for c in range(CONTEXT_SIZE + 1):
        ax[0][c].imshow(nom["obs"][c]); ax[1][c].imshow(cf["obs"][c])
    ax[0][-1].imshow(nom["goal"]); ax[1][-1].imshow(cf["goal"])
    for r in (0, 1):
        for c in range(ncol):
            ax[r][c].set_xticks([]); ax[r][c].set_yticks([])
    ax[0][0].set_ylabel("nominal\nobs → goal", rotation=0, ha="right", va="center", fontsize=9)
    ax[1][0].set_ylabel("counterfactual\n(NWM) → goal", rotation=0, ha="right", va="center", fontsize=9)
    for c in range(2, ncol):
        ax[2][c].axis("off")
    gs = ax[2][0].get_gridspec()
    for c in range(2):
        ax[2][c].remove()
    axw = fig.add_subplot(gs[2, :2])
    axw.plot(nom["action"][:, 0], nom["action"][:, 1], "k-o", ms=4, label="nominal label")
    axw.plot(cf["action"][:, 0], cf["action"][:, 1], "r--o", ms=4, label="recovery label")
    axw.set_aspect("equal"); axw.set_title("waypoint labels (local frame, m)"); axw.legend(fontsize=8)
    fig.suptitle("%s  t=%s" % (str(cf["key"]), str(cf["t"])))
    plt.tight_layout(); plt.show(); plt.close(fig)


show_clip()
show_pair()

## 9. Dataset + ViNT policy + training

Trains on whatever is in `CACHE_DIR`. Set `PULL_FULL_DATASET = True` to first
pull the whole accumulated dataset from the HF dataset repo.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from vint_train.models.vint.vint import ViNT

PULL_FULL_DATASET = False
if PULL_FULL_DATASET:
    from huggingface_hub import snapshot_download
    snapshot_download(DATA_REPO_ID, repo_type="dataset", local_dir=CACHE_DIR,
                      token=HF_TOKEN, allow_patterns=["*.npz"])

_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


class BCCacheDataset(Dataset):
    def __init__(self, split, train_aug):
        self.files = sorted(glob.glob(os.path.join(CACHE_DIR, split, "**", "*.npz"), recursive=True))
        self.train_aug = train_aug

    def __len__(self):
        return len(self.files)

    def _aug(self, imgs):
        s = random.uniform(0.85, 1.0)
        H, W = IMAGE_SIZE[1], IMAGE_SIZE[0]
        ch, cw = int(round(H * s)), int(round(W * s))
        top, left = random.randint(0, H - ch), random.randint(0, W - cw)
        bri, con = random.uniform(0.8, 1.2), random.uniform(0.8, 1.2)
        out = []
        for im in imgs:
            im = cv2.resize(im[top:top + ch, left:left + cw], IMAGE_SIZE, interpolation=cv2.INTER_LINEAR).astype(np.float32)
            out.append(np.clip((im - 128) * con + 128 * bri, 0, 255))
        return out

    def _norm(self, im):
        t = torch.from_numpy(np.ascontiguousarray(im)).permute(2, 0, 1).float() / 255.0
        return (t - _MEAN) / _STD

    def __getitem__(self, i):
        d = np.load(self.files[i])
        obs, goal = list(d["obs"]), d["goal"]
        if self.train_aug:
            aug = self._aug(obs + [goal])
            obs, goal = aug[:-1], aug[-1]
        obs_t = torch.cat([self._norm(o) for o in obs], dim=0)
        return obs_t, self._norm(goal), torch.from_numpy(d["action"].astype(np.float32)), torch.tensor(float(d["dist"]))


train_ds = BCCacheDataset("train", AUG)
val_ds = BCCacheDataset("test", False)
print("train %d | val %d" % (len(train_ds), len(val_ds)))
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
                      pin_memory=True, drop_last=len(train_ds) > BATCH_SIZE)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

vint = ViNT(context_size=CONTEXT_SIZE, len_traj_pred=LEN_TRAJ_PRED, learn_angle=True,
            obs_encoder="efficientnet-b0", obs_encoding_size=VINT_OBS_ENCODING_SIZE, late_fusion=False,
            mha_num_attention_heads=VINT_MHA, mha_num_attention_layers=VINT_MHA, mha_ff_dim_factor=VINT_MHA)
try:
    from efficientnet_pytorch import EfficientNet
    _m, _u = vint.obs_encoder.load_state_dict(EfficientNet.from_pretrained("efficientnet-b0").state_dict(), strict=False)
    print("obs_encoder ImageNet init (%d/%d missing/unexpected)" % (len(_m), len(_u)))
except Exception as e:
    print("ImageNet init skipped:", e)

VINT_CONFIG = dict(context_size=CONTEXT_SIZE, len_traj_pred=LEN_TRAJ_PRED, learn_angle=True,
                   obs_encoder="efficientnet-b0", obs_encoding_size=VINT_OBS_ENCODING_SIZE, late_fusion=False,
                   mha_num_attention_heads=VINT_MHA, mha_num_attention_layers=VINT_MHA, mha_ff_dim_factor=VINT_MHA,
                   image_size=list(IMAGE_SIZE), goal_horizon=GOAL_HORIZON, waypoint_scale=WAYPOINT_SCALE,
                   step_offsets=STEP_OFFSETS.tolist(), perturbation_levels=PERTURBATION_LEVELS,
                   counterfactual_source="nwm-cdit-%s" % NWM_SIZE, data_repo=DATA_REPO_ID)

if RESUME_FROM_HF:
    try:
        from huggingface_hub import hf_hub_download as _dl
        from safetensors.torch import load_file as _lsf
        _rid = "%s/%s" % (HF_USER, REPO_NAME)
        vint.load_state_dict(_lsf(_dl(_rid, "model.safetensors", token=HF_TOKEN)), strict=True)
        print("warm-started from", _rid)
    except Exception as e:
        print("no HF warm-start:", e)
vint = vint.to(device)

In [ ]:
def compute_loss(dp, ap, al, dl, alpha=ALPHA):
    dloss = F.mse_loss(dp.squeeze(-1), dl)
    aloss = F.mse_loss(ap, al)
    with torch.no_grad():
        wc = F.cosine_similarity(ap[..., :2], al[..., :2], dim=-1).mean()
    return (1 - alpha) * aloss + alpha * 1e-2 * dloss, aloss.detach(), wc


opt = torch.optim.AdamW(vint.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
spe = max(1, len(train_dl))
tot_steps, warm = EPOCHS * spe, WARMUP_EPOCHS * spe
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: s / max(1, warm) if s < warm else 0.5 * (1 + math.cos(math.pi * min(1.0, (s - warm) / max(1, tot_steps - warm)))))
scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and device.type == "cuda"))


@torch.no_grad()
def evaluate():
    vint.eval()
    tot = cos = n = 0.0
    for obs, goal, act, dist in val_dl:
        obs, goal, act, dist = obs.to(device), goal.to(device), act.to(device), dist.to(device)
        with torch.autocast(device_type=device.type, enabled=(USE_AMP and device.type == "cuda")):
            dp, ap = vint(obs, goal)
            t, _, wc = compute_loss(dp, ap, act, dist)
        b = obs.size(0); tot += t.item() * b; cos += wc.item() * b; n += b
    return (tot / n, cos / n) if n else (float("nan"), float("nan"))


@torch.no_grad()
def rollout_plot(ep):
    vint.eval()
    obs, goal, act, _ = next(iter(val_dl if len(val_ds) else train_dl))
    _, ap = vint(obs.to(device), goal.to(device))
    ap, act = ap.float().cpu().numpy(), act.numpy()
    m = min(4, obs.size(0))
    fig, ax = plt.subplots(1, m, figsize=(4 * m, 4), squeeze=False)
    for j in range(m):
        ax[0][j].plot(act[j, :, 0], act[j, :, 1], "k-o", ms=3, label="label")
        ax[0][j].plot(ap[j, :, 0], ap[j, :, 1], "r--o", ms=3, label="pred")
        ax[0][j].set_aspect("equal")
        if j == 0:
            ax[0][j].legend()
    fig.suptitle("epoch %d" % ep); plt.show(); plt.close(fig)


assert len(train_ds) > 0, "empty training cache -- run Section 8, or set PULL_FULL_DATASET"
best = float("inf"); step = 0
for ep in range(EPOCHS):
    vint.train()
    run = defaultdict(float); seen = 0
    for obs, goal, act, dist in train_dl:
        obs, goal, act, dist = obs.to(device), goal.to(device), act.to(device), dist.to(device)
        with torch.autocast(device_type=device.type, enabled=(USE_AMP and device.type == "cuda")):
            dp, ap = vint(obs, goal)
            loss, aloss, wc = compute_loss(dp, ap, act, dist)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); step += 1
        b = obs.size(0); seen += b
        run["t"] += loss.item() * b; run["a"] += aloss.item() * b; run["c"] += wc.item() * b
    seen = max(seen, 1)
    vt, vc = evaluate()
    crit = vt if not math.isnan(vt) else run["t"] / seen
    print("ep %3d | lr %.2e | train %.4f (act %.4f wpcos %.3f) | val %.4f (wpcos %.3f)"
          % (ep, sched.get_last_lr()[0], run["t"] / seen, run["a"] / seen, run["c"] / seen, vt, vc))
    if ep % 10 == 0 or ep == EPOCHS - 1:
        rollout_plot(ep)
    if crit < best:
        best = crit
        torch.save({"model": vint.state_dict(), "config": VINT_CONFIG, "epoch": ep,
                    "val_total": float(vt), "val_wp_cos": float(vc)}, os.path.join(CKPT_DIR, "best.pt"))
print("best:", best)

## 10. Publish the policy

In [ ]:
from huggingface_hub import upload_folder
from safetensors.torch import save_file

repo_id = "%s/%s" % (HF_USER, REPO_NAME)
create_repo(repo_id, private=True, exist_ok=True, token=HF_TOKEN, repo_type="model")
EXP = "/content/hf_export"
shutil.rmtree(EXP, ignore_errors=True); os.makedirs(EXP)

bestc = torch.load(os.path.join(CKPT_DIR, "best.pt"), map_location="cpu")
save_file(bestc["model"], os.path.join(EXP, "model.safetensors"))

manifest = load_manifest()
cfg = dict(VINT_CONFIG, trained_from_scratch=True, obs_encoder_init="imagenet",
           n_trajectories_processed=len([k for k in manifest if "error" not in manifest[k]]),
           n_train_samples=len(glob.glob(os.path.join(CACHE_DIR, "train", "**", "*.npz"), recursive=True)),
           n_val_samples=len(glob.glob(os.path.join(CACHE_DIR, "test", "**", "*.npz"), recursive=True)),
           best_val_total_loss=bestc["val_total"], best_val_wp_cos=bestc["val_wp_cos"], best_epoch=bestc["epoch"])
with open(os.path.join(EXP, "config.json"), "w") as f:
    json.dump(cfg, f, indent=2)

card = "\n".join([
    "---", "license: cc-by-nc-4.0",
    "tags: [robotics, visual-navigation, behavior-cloning, vint, sacson, world-model]", "---", "",
    "# ViNT-SACSoN corrective policy (NWM counterfactuals)", "",
    "ViNT goal-conditioned navigation policy **trained from scratch** on corrective",
    "behaviour-cloning data. Drift-then-recover history frames are generated by the",
    "[Navigation World Model](https://huggingface.co/facebook/nwm) (CDiT-%s)" % NWM_SIZE,
    "conditioned on a perturbed action sequence; nominal samples keep the real",
    "[SACSoN/HuRoN](https://sites.google.com/view/sacson-review) frames. Poses are",
    "MBRA-relabelled. Training data: `%s`." % cfg["data_repo"], "",
    "Inputs: `obs_img [B,%d,%d,%d]`, `goal_img [B,3,%d,%d]` -> `(dist, action[B,%d,4])`"
    % (3 * (CONTEXT_SIZE + 1), IMAGE_SIZE[1], IMAGE_SIZE[0], IMAGE_SIZE[1], IMAGE_SIZE[0], LEN_TRAJ_PRED),
    "(cumulative local waypoints x,y,cos,sin in metres).", "",
    "```python",
    "from vint_train.models.vint.vint import ViNT",
    "import json, safetensors.torch as st",
    "c = json.load(open('config.json'))",
    "k = ['context_size','len_traj_pred','learn_angle','obs_encoder','obs_encoding_size',",
    "     'late_fusion','mha_num_attention_heads','mha_num_attention_layers','mha_ff_dim_factor']",
    "m = ViNT(**{x: c[x] for x in k}); m.load_state_dict(st.load_file('model.safetensors')); m.eval()",
    "```", "",
    "## Limitations",
    "- Counterfactual frames are **NWM generations**, not real observations (domain gap;",
    "  NWM trained on a private higher-res SACSoN).",
    "- Poses MBRA-estimated; `dt` calibrated from the dataset `position` attr.",
    "- SACSoN spans 3 buildings -- weak out-of-distribution transfer.",
    "- CC-BY-NC 4.0 (inherited from NWM).", "",
    "Best epoch %d: val loss %.4f, val waypoint cos-sim %.3f."
    % (bestc["epoch"], bestc["val_total"], bestc["val_wp_cos"]),
])
with open(os.path.join(EXP, "README.md"), "w") as f:
    f.write(card)

upload_folder(folder_path=EXP, repo_id=repo_id, token=HF_TOKEN, repo_type="model")
print("model  ->", "https://huggingface.co/" + repo_id)
print("data   ->", "https://huggingface.co/datasets/" + DATA_REPO_ID)
print("manifest ->", MANIFEST_PATH)

## 11. Notes

- The sample cache (`/content/bc_cache`) is **not wiped** — it mirrors the HF
  dataset repo. Re-run Section 8 in new sessions to process more trajectories
  (manifest-resumed), then Section 9–10 to retrain on the larger set
  (`PULL_FULL_DATASET = True` to pull everything first).
- Speed knobs: `NWM_SIZE`, `NWM_DDIM_STEPS`, `NWM_BATCH`, `MAX_SAMPLES_PER_TRAJ`,
  `SAMPLE_STRIDE`.
- First-run checks: eyeball a `counterfactual_clips/<key>/` rollout (drift should
  be visible and coherent); if frames look wrong, the likely culprits are the
  NWM action normalization (`METRIC_WAYPOINT_SPACING`), `rel_t` (currently
  `step/128`), or the EMA-key name in the checkpoint.

In [ ]:
print("nothing to clean up -- data kept in", CACHE_DIR, "and", DATA_REPO_ID)